### README

These notebooks contain the instructions and code to run the models with our datasets. They cover two kinds of work: **decomposition notebooks** (PCA, ICA, SVD, ZCA) that look at the structure of the spectra without training a classifier, and **classification notebooks** (Random Forest, XGBoost, CNN, Transformer) that train and evaluate a model.

##### Data

The datasets are stored in the *data* folder. There are two subfolders containing csv-format datasets for mixed ratio and single species replicates.

- **data/mixed_ratio** — Staph : PAO mixtures at five ratios (1:9, 3:7, 5:5, 7:3, 9:1). Each notebook picks the class off the file name, so `1-9_S-P_1.csv` becomes `1:9 | Staph : PAO`.
- **data/six_single_species** — six single-species replicates: Corn, E.coli, FAE, Malt, PAO1, STAPH.

In every csv the **first row is the wavenumber axis** and each row after it is one spectrum. The *raw_data* folder keeps the original `.mat` acquisitions the csv files were exported from.

`combine_csvs_by_prefix.py` is a helper for stacking several csv files of the same species into one `<species>_final.csv`, using the same file-name matching the notebooks use (it keeps the header row of the first file and skips it for the rest):

```
python combine_csvs_by_prefix.py [input_folder] [-o OUTPUT_FOLDER]
```

##### Models

There are pre-trained models in the *saved_models* folder and these models are the ones trained directly from the notebooks provided.

| File | Notebook that produced it |
| --- | --- |
| `rf_mixed_model.pkl` | RF_mixed.ipynb |
| `rf_single_model.pkl` | RF_six_single.ipynb |
| `xg_mixed_model.pkl` | XG_mixed.ipynb |
| `xg_single_model.pkl` | XG_six_single.ipynb |
| `cnn_mixed.keras` | CNN_mixed.ipynb |
| `cnn_six_single.keras` | CNN_six_single.ipynb |
| `transformer_mixed.keras` | Transformer_mixed.ipynb |
| `transformer_six_single.keras` | Transformer_six_single.ipynb |

The `.pkl` files reload with `joblib.load(path)`; the `.keras` files with `tensorflow.keras.models.load_model(path)`.

#### The Notebooks

| Notebook | Type | Dataset | What it produces |
| --- | --- | --- | --- |
| PCA_mixed | decomposition | mixed ratio | PC score plot + loadings, explained variance |
| ICA_mixed | decomposition | mixed ratio | independent component spectra, per-class IC heatmap |
| SVD_mixed | decomposition | mixed ratio + pure species | singular vectors, variance explained, `svd_visual.png` |
| ZCA_mixed | decomposition | mixed ratio | whitened spectra on the most group-separating variables |
| RF_mixed / RF_six_single | classifier | mixed ratio / six species | trained `.pkl`, accuracy, F1, confusion matrix, CV, learning curve |
| XG_mixed / XG_six_single | classifier | mixed ratio / six species | trained `.pkl`, accuracy, F1, confusion matrix, CV, learning curve |
| CNN_mixed / CNN_six_single | classifier | mixed ratio / six species | trained `.keras`, ROC/AUC, confusion matrix, CV |
| Transformer_mixed / Transformer_six_single | classifier | mixed ratio / six species | trained `.keras`, ROC/AUC, confusion matrix, CV |

Every notebook opens with the same three steps, so once you have read one loader you have read them all:

1. **Load Libraries.**
2. **Load and Add Target Columns.** Reads every csv in the data folder, assigns a class by matching the `targets` dictionary against the file name, label-encodes that class, and writes `target_encoding_map.json` so you can map an encoded value back to a species.
3. **Label Data and Drop Target Column.** Renames the wavenumber columns to `1 … N` and writes the original wavenumber for each one to `feature_map.json`, then drops the text target column (the encoded one stays).

Both json files are rewritten every run, so they always describe the data currently loaded.

#### Decomposition Notebooks

These four do not train a classifier. They reduce or whiten the spectra and plot the result, to show how separable the classes are and which Raman shifts drive that separation. All four standardize the features with `StandardScaler` before decomposing, so no single high-intensity band dominates just because of its scale.

##### PCA — `PCA_mixed.ipynb`
Principal component analysis on the five mixed-ratio classes.

- The loader here also **interpolates every file onto one shared wavenumber axis** (`np.interp`), so files acquired on slightly different axes still line up column for column.
- `run_pca_spectral_style()` fits `sklearn.decomposition.PCA` with `n_components=10` and prints the explained variance for PC1–PC5 plus the PC1+PC2 total.
- The figure has two panels: a **PC1 vs PC2 score plot** with its own marker and colour per class, and a **top-15 loading plot** showing which wavenumbers push samples along those axes.
- It returns a dictionary with the score dataframe, the fitted `PCA` object, the scaler, the raw scores, loadings, explained-variance ratios and feature column names, so you can carry the fit into your own cell.
- Set `five_points_per_group=True` to plot only the first five samples per class when the full scatter is too crowded.

##### ICA — `ICA_mixed.ipynb`
Independent component analysis on the same five classes. Where PCA looks for directions of maximum variance, ICA looks for statistically independent source signals — closer to asking which underlying spectra were mixed together.

- `fit_fast_ica()` runs `FastICA` with `n_components=5`, `whiten="unit-variance"`, the `logcosh` contrast function, `random_state=42`, `max_iter=5000` and `tol=1e-4`.
- Components come out of FastICA in arbitrary order and sign, so the notebook **reorders them by absolute excess kurtosis** (most non-Gaussian, i.e. most source-like, first) and **flips each one's sign** so its largest peak points up. That makes reruns comparable.
- It reads `feature_map.json` back in to plot the components against real Raman shift rather than column number.
- The figure is 2×2: samples projected onto IC1/IC2, samples on IC1 against another component, a **heatmap of mean IC score per class**, and the **recovered independent component spectra** with each one's kurtosis in the legend.

##### SVD — `SVD_mixed.ipynb`
Singular value decomposition, computed directly with `np.linalg.svd` rather than through a scikit-learn estimator.

- **This is the one notebook that loads more than the mixed ratios.** It reads `data/mixed_ratio` *and* the pure single-species spectra, adding two endmember classes — `10:0 | Staph : PAO` (from `STAPH` files) and `0:10 | Staph : PAO` (from `PAO1` files) — for seven classes total, so the mixtures can be seen against the two things being mixed.
- `run_svd()` keeps 10 components, forms the scores as `U · Σ`, and computes each component's explained variance against the **full** singular spectrum (not just the retained 10). It prints SV1–SV5 and the SV1+SV2 total.
- The figure is 2×2: SV1 vs SV2 scores, a variance-explained plot, the top-15 spectral contributors, and the right singular vectors plotted against Raman shift.
- This notebook **saves `svd_visual.png` at 300 dpi** next to itself.

##### ZCA — `ZCA_mixed.ipynb`
Zero-phase component analysis whitening on the five mixed-ratio classes.

- ZCA differs from the other three in an important way: **it is not a dimensionality reduction.** It decorrelates the features and sets them to unit variance while rotating back into the original feature space, so every wavenumber is still a variable afterwards and the whitened data stays interpretable band by band.
- `fit_transform_zca()` builds the whitening matrix from the covariance eigendecomposition (`np.linalg.eigh`) as `U · diag(1/√(λ + ε)) · Uᵀ`, with `epsilon=1e-5` guarding against tiny eigenvalues blowing up.
- Because no component ordering falls out of ZCA, `group_separation_scores()` ranks the whitened variables by between-group over within-group variance and the **two best-separating variables become the plot axes**.
- The figure has two panels: the whitened scatter on those two variables, and the top-15 variables by separation score.
- The loader is set up to read the single-species folder as well, but that line is commented out, so as shipped it runs on the five mixed ratios only. Uncomment it to include the pure species.

#### Classification Notebooks

Each model comes in two versions: `*_mixed` trains on the five Staph : PAO ratios in `data/mixed_ratio`, and `*_six_single` trains on the six species in `data/six_single_species`. Apart from the dataset and the saved file name the two versions are the same notebook.

All four models use the same **80/20 stratified train/test split with `random_state=42`**, so results are reproducible and comparable across models.

##### Random Forest — `RF_mixed.ipynb`, `RF_six_single.ipynb`

- **§4 Training.** `train_rf_model()` does the split and fits a `RandomForestClassifier`. By default it uses fixed, already-tuned hyperparameters: `n_estimators=200`, `max_depth=10`, `min_samples_split=5`, `min_samples_leaf=2`, `max_features="sqrt"` — depth and leaf size are capped to keep the forest from simply memorising the training spectra.
- Set `use_optuna=True` to run a 50-trial Optuna search instead of the fixed values. Optuna is imported defensively, so the notebook still runs without it installed as long as you leave tuning off.
- `save_model=True` writes the fitted forest to `saved_models/rf_mixed_model.pkl` (or `rf_single_model.pkl`).
- **§5 Evaluation.** `eval_model()` reports accuracy, weighted F1, a confusion matrix plot and a full classification report. Pass `export_class_report=True` to also write the report to csv.
- **§6 Run.** Trains, then prints train vs test accuracy and the gap between them, runs 5-fold `cross_val_score`, plots a `learning_curve` over increasing training-set size, and finally evaluates on the held-out 20%.

##### XGBoost — `XG_mixed.ipynb`, `XG_six_single.ipynb`

- Same layout as the Random Forest notebooks: `train_xgboost_model()` in §4, `eval_model()` in §5, and the same train-vs-test gap, 5-fold cross-validation, learning curve and final evaluation in §6.
- Defaults to a fixed set of already-tuned boosting parameters (tree depth, learning rate, subsample and colsample fractions, min child weight, gamma and the L1/L2 regularization terms). `use_optuna=True` runs a 50-trial search over them instead.
- `save_model=True` writes `saved_models/xg_mixed_model.pkl` (or `xg_single_model.pkl`).
- Worth running alongside the forest: they take the same inputs and the same evaluation, so the numbers are directly comparable.

##### CNN — `CNN_mixed.ipynb`, `CNN_six_single.ipynb`
A 1-D convolutional network that reads each spectrum as a sequence, letting the filters pick up local band shapes rather than treating every wavenumber as an unrelated feature.

- **§4 Splitting.** Standardizes the features, one-hot encodes the labels, does the 80/20 stratified split, and reshapes to `(samples, timesteps, 1)` for `Conv1D`. Shape and NaN checks run here so a misaligned csv fails loudly instead of training on garbage.
- **§5 Architecture + tuning.** A stack of `Conv1D → BatchNormalization → LeakyReLU → MaxPooling1D` blocks feeding `Flatten → Dropout → Dense → softmax`. Optuna's TPE sampler tunes the number of conv blocks, filter count, kernel and pool size, dense width, dropout, L2 strength, LeakyReLU slope, learning rate and batch size. Defaults are `OPTUNA_N_TRIALS = 5` with seed 42 and `MAX_EPOCHS = 50` — **raise the trial count for a real search**, it is set low so the notebook finishes quickly out of the box.
- Training uses `EarlyStopping` plus a custom callback that halts a run as soon as validation loss starts climbing.
- **§6 Training** on the winning parameters, then the model is saved to `saved_models/cnn_mixed.keras` (or `cnn_six_single.keras`).
- **§6b** plots per-epoch training vs validation accuracy and loss — a widening gap is the overfitting signal.
- **§7** confusion matrix, classification report, and one-vs-rest ROC curves with AUC per class. **§7b** prints the train/test accuracy gap.
- **§8** 5-fold `StratifiedKFold` cross-validation. Each fold rebuilds and retrains the network from scratch and refits the `StandardScaler` on that fold's training rows only, so nothing leaks from validation into preprocessing. This is the slow cell — it trains the network five more times.

##### Transformer — `Transformer_mixed.ipynb`, `Transformer_six_single.ipynb`
The same experiment as the CNN with self-attention in place of convolution: the spectrum is cut into patches and attention lets distant bands inform each other directly, instead of only through stacked local filters.

- Section layout is identical to the CNN notebooks (§4 splitting, §5 architecture + Optuna, §6 training, §6b learning curves, §7 performance, §7b train/test gap, §8 5-fold cross-validation), so the two are directly comparable.
- **§5** builds patch embeddings followed by multi-head self-attention blocks. Optuna tunes `patch_size` (4/8/16), `d_model` (32/64/128), `num_heads` (2/4/8), `ff_dim` (64/128/256), the number of transformer blocks (1–4), MLP width, dropout, L2 strength, learning rate and batch size. `OPTUNA_N_TRIALS` is again 5 by default — raise it for a real search.
- The trained model is saved to `saved_models/transformer_mixed.keras` (or `transformer_six_single.keras`).
- This is the heaviest notebook of the eight; run it on the GPU.

#### Getting Started On Your Own

##### Random Forest and XG Boost Models
These models can be trained and ran in a virtual python environment, or directly in your global python environment. However, you need to install certain libraries and these libraries are provided in *requirements.txt*. First install the libraries in either environment, then simply open and run the notebooks in that same environment.

The same applies to the four decomposition notebooks (PCA, ICA, SVD, ZCA) — they only need numpy, pandas, scikit-learn, matplotlib and seaborn, so they run anywhere the forest runs, and they finish in seconds.

```
pip install -r requirements.txt
```

##### Convolutional Neural Networks (CNN) and Transformers
These models can be trained in a virtual or global python environment, but to use your GPU to train, you must install Docker as a prerequisite. Once docker is installed, use the following command in your command line interface (CLI) to run the notebooks. Once the notebook is finished running, the model will be trained, evaluated, and saved.

In [ ]:
.\train_gpu_headless.ps1 <file name>

The `.ipynb` extension is optional, so `.\train_gpu_headless.ps1 CNN_mixed` works too. The script runs the notebook in place with `nbconvert`, so every plot and printed metric is saved back into the notebook file and the trained model lands in `saved_models\`.

The script lives in `New Notebooks\`, next to the notebooks it runs. A forwarder in this folder hands off to it, so the same command works from here too.

Only the `CNN_*` and `Transformer_*` notebooks need Docker, since they are the ones that use the GPU. The image has no xgboost, so the script refuses `XG_*` notebooks; run those (and the CPU-only `RF_*` ones) in a normal Python environment.

Note that pip-installed TensorFlow is CPU-only on native Windows (GPU support was dropped after TF 2.10). The Docker image the script uses bundles a GPU-capable build, which is why it exists.

##### A Note on Data Paths
Every notebook in `New Notebooks\` uses the same relative paths: `..\data\mixed_ratio` or `..\data\six_single_species` for the spectra, and `..\saved_models` for the trained model. Those point one level **up**, at the `data\` and `saved_models\` folders sitting beside `New Notebooks\`.

That single convention is what lets the Docker runner work without edits. `train_gpu_headless.ps1` mounts `Distributable Notebooks\` at `/workspace` and runs from `/workspace/New Notebooks`, so `..\data` resolves to the same folder inside the container as it does on Windows. Opening the notebooks directly in Jupyter from inside `New Notebooks\` behaves identically, so no path editing is needed either way.